In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
%cd /content/drive/MyDrive/연세_수업자료/2학기/텍스트이해와인공지능/기말과제

/content/drive/MyDrive/연세_수업자료/2학기/텍스트이해와인공지능/기말과제


In [ ]:
!pip install transformers
!pip install torchinfo
!pip install pytorch-lightning
!pip install torchmetrics
!pip instlal sentenecepiece

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 776.9/776.9 kB 7.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 805.2/805.2 kB 15.4 MB/s eta 0:00:00
ERROR: unknown command "instlal" - maybe you meant "install"


In [ ]:
import numpy as np
import pandas as pd
import os, sys, re, math, random, time, json, pickle, tqdm, gc
from collections import defaultdict
import itertools as it

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader

from transformers import AdamW, get_linear_schedule_with_warmup
from transformers.optimization import get_cosine_schedule_with_warmup
from transformers import AutoTokenizer, AutoModelForSequenceClassification

from sklearn.model_selection import StratifiedKFold, KFold
from sklearn.metrics import f1_score, accuracy_score
from sklearn import preprocessing
from sklearn.model_selection import train_test_split
from tqdm import tqdm

class args:
    gpu = '0'
    label_size = 2
    epochs = 10
    batch_size = 32
    weight_decay = 0.01
    max_len = 100
    num_workers = 0
    seed = 2022
    warmup_steps = 200
    logging_steps = 10
    evaluation_strategy = "epoch"
    start_lr = 2e-5

os.environ["CUDA_VISIBLE_DEVICES"] = '0'
device = torch.device(f"cuda" if torch.cuda.is_available() else "cpu")
print(device)

path = '/content/drive/MyDrive/연세_수업자료/2학기/텍스트이해와인공지능/'

df_test = pd.read_csv('/content/drive/MyDrive/연세_수업자료/2학기/텍스트이해와인공지능/기말과제/df_test.csv', encoding = 'utf-8')
df_train_pre = pd.read_csv('/content/drive/MyDrive/연세_수업자료/2학기/텍스트이해와인공지능/df_train_pre.csv', encoding = 'utf-8')
df_dev_pre = pd.read_csv('/content/drive/MyDrive/연세_수업자료/2학기/텍스트이해와인공지능/df_dev_pre.csv', encoding = 'utf-8')
df_test_pre = pd.read_csv('/content/drive/MyDrive/연세_수업자료/2학기/텍스트이해와인공지능/df_test_pre.csv', encoding = 'utf-8')

df_train_pre.dropna(inplace = True)
df_dev_pre.dropna(inplace = True)
df_test_pre.dropna(inplace = True)

df_train_pre.reset_index(drop = True, inplace = True)
df_dev_pre.reset_index(drop = True, inplace = True)

cuda


In [ ]:
from tokenizers.processors import TemplateProcessing

class CustomDataset(Dataset):
    def __init__(self, dataset, text, label, tokenizer, max_len):
        self.dataset = dataset
        self.text = text
        self.label = label
        self.tokenizer = tokenizer
        self.max_len = max_len

    def __getitem__(self, idx):
        text = self.dataset.iloc[idx]['input']
        label = self.dataset.iloc[idx]['output']

        inputs = self.tokenizer(text,
                                return_tensors='pt',
                                truncation=True,
                                max_length=self.max_len,
                                padding='max_length',
                                add_special_tokens=True)

        input_ids = inputs['input_ids'][0]
        attention_mask = inputs['attention_mask'][0]

        return input_ids, attention_mask, label

    def __len__(self):
        return len(self.dataset)

model_name = "beomi/kcbert-base"
tokenizer = AutoTokenizer.from_pretrained(model_name)

train_dataset = CustomDataset(df_train_pre, 'input', 'output', tokenizer, args.max_len)
train_loader = DataLoader(train_dataset, batch_size=args.batch_size, shuffle=True)
valid_dataset = CustomDataset(df_dev_pre, 'input', 'output', tokenizer, args.max_len)
valid_loader = DataLoader(valid_dataset, batch_size=args.batch_size, shuffle=False)

tokenizer_config.json:   0%|          | 0.00/49.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/619 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/250k [00:00<?, ?B/s]

In [ ]:
model = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=args.label_size)
model.to(device)

model.config.pad_token_id = tokenizer.pad_token_id

loss_fn = nn.CrossEntropyLoss()
optimizer = AdamW(model.parameters(), lr=args.start_lr)
scheduler = get_linear_schedule_with_warmup(optimizer,
                                            num_warmup_steps = int(len(train_loader)*args.epochs * 0.1),
                                            num_training_steps = len(train_loader)*args.epochs) # cosine annealing

def train(model, data_loader, loss_fn, optimizer, scheduler):
    train_loss = 0
    target_lst = []
    pred_lst = []
    i = 0
    model.train()
    pbar = tqdm(data_loader)
    for ids, mask, target in pbar:
        i += 1
        ids = ids.to(device)
        mask = mask.to(device)
        target = target.to(device)

        optimizer.zero_grad()
        output = model(ids, mask)
        loss = loss_fn(output.logits, target).to(device)
        train_loss += loss.item()

        loss.backward()
        optimizer.step()
        scheduler.step()

        target_lst.extend(target.detach().cpu().numpy())
        pred_lst.extend(output.logits.argmax(dim=1).tolist())
        pbar.set_description('\033[1m[C_loss : {:>.5}]\033[0m'.format(round(train_loss / i, 4)))
    train_loss = train_loss / len(data_loader)
    train_score = f1_score(y_true=target_lst, y_pred=pred_lst, average='macro')
    train_acc = accuracy_score(y_true=target_lst, y_pred=pred_lst)

    torch.cuda.empty_cache()
    gc.collect()

    return model, train_loss, train_score, train_acc

def valid(model, data_loader, loss_fn):
    valid_loss = 0
    target_lst = []
    pred_lst = []
    i = 0
    model.eval()
    pbar = tqdm(data_loader)
    for ids, mask, target in pbar:
        i += 1
        ids = ids.to(device)
        mask = mask.to(device)
        target = target.to(device)

        output = model(ids, mask)
        loss = loss_fn(output.logits, target).to(device)
        valid_loss += loss.item()

        target_lst.extend(target.detach().cpu().numpy())
        pred_lst.extend(output.logits.argmax(dim=1).tolist())
        pbar.set_description('\033[1m[C_loss : {:>.5}]\033[0m'.format(round(valid_loss / i, 4)))
    valid_loss = valid_loss / len(data_loader)
    valid_score = f1_score(y_true=target_lst, y_pred=pred_lst, average='macro')
    valid_acc = accuracy_score(y_true=target_lst, y_pred=pred_lst)

    torch.cuda.empty_cache()
    gc.collect()

    return model, valid_loss, valid_score, valid_acc

stop_val_f1 = 0
stop_count = 0

for epoch in range(args.epochs):
    if stop_count == 3:
        break

    model, train_loss, train_score, train_acc = train(model, train_loader, loss_fn, optimizer, scheduler)
    model, valid_loss, valid_score, valid_acc = valid(model, valid_loader, loss_fn)

    print(f'Epoch : {epoch + 1},    t_loss : {round(train_loss, 4)},   t_f1 : {round(train_score, 4)},   t_acc : {round(train_acc, 4)}\n')
    print(f'              v_loss : {round(valid_loss, 4)},   v_f1 : {round(valid_score, 4)},   v_acc : {round(valid_acc, 4)}\n')

    if valid_score > stop_val_f1:
        default_weight_path = path + 'ddd_kcbert_base_e.pt'
        torch.save(model.state_dict(), default_weight_path)
        stop_val_f1 = valid_score
        stop_count = 0
    else:
        stop_count += 1

    torch.cuda.empty_cache()
    gc.collect()

# gpu 사용: 4.7gb

model.safetensors:   0%|          | 0.00/438M [00:00<?, ?B/s]

Some weights of BertForSequenceClassification were not initialized from the model checkpoint at beomi/kcbert-base and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
/usr/local/lib/python3.10/dist-packages/transformers/optimization.py:411: FutureWarning: This implementation of AdamW is deprecated and will be removed in a future version. Use the PyTorch implementation torch.optim.AdamW instead, or set `no_deprecation_warning=True` to disable this warning
  warnings.warn(
[C_loss : 0.2084]: 100%|██████████| 65/65 [00:12<00:00,  5.06it/s]


Epoch : 1,    t_loss : 0.3588,   t_f1 : 0.824,   t_acc : 0.8315

              v_loss : 0.2084,   v_f1 : 0.9198,   v_acc : 0.9241



[C_loss : 0.1938]: 100%|██████████| 65/65 [00:12<00:00,  5.14it/s]


Epoch : 2,    t_loss : 0.1598,   t_f1 : 0.9407,   t_acc : 0.9429

              v_loss : 0.1938,   v_f1 : 0.922,   v_acc : 0.9241



...

Epoch : 8,    t_loss : 0.003,   t_f1 : 0.9991,   t_acc : 0.9991

              v_loss : 0.3862,   v_f1 : 0.927,   v_acc : 0.9294



[C_loss : 0.4255]: 100%|██████████| 65/65 [00:12<00:00,  5.31it/s]


Epoch : 9,    t_loss : 0.0016,   t_f1 : 0.9994,   t_acc : 0.9995

              v_loss : 0.4255,   v_f1 : 0.9301,   v_acc : 0.9323



[C_loss : 0.4289]: 100%|██████████| 65/65 [00:12<00:00,  5.32it/s]


Epoch : 10,    t_loss : 0.0012,   t_f1 : 0.9996,   t_acc : 0.9996

              v_loss : 0.4289,   v_f1 : 0.9285,   v_acc : 0.9309



In [ ]:
class TestDataset(Dataset):

    def __init__(self, dataset, text, tokenizer, max_len):
        self.dataset = dataset
        self.text = text
        self.tokenizer = tokenizer
        self.max_len = max_len

    def __getitem__(self, idx):
        text = self.dataset.iloc[idx]['input']
        inputs = self.tokenizer(text,
                                return_tensors='pt',
                                truncation=True,
                                max_length=self.max_len,
                                padding='max_length',
                                add_special_tokens=True)

        input_ids = inputs['input_ids'][0]
        attention_mask = inputs['attention_mask'][0]

        return input_ids, attention_mask

    def __len__(self):
        return len(self.dataset)

model_name = "beomi/kcbert-base"
tokenizer = AutoTokenizer.from_pretrained(model_name)
test_dataset = TestDataset(df_test_pre, 'input', tokenizer, args.max_len)
test_loader = DataLoader(test_dataset, batch_size=args.batch_size, shuffle=False)

tokenizer_config.json:   0%|          | 0.00/49.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/619 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/250k [00:00<?, ?B/s]

In [ ]:
model = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=args.label_size)
model.config.pad_token_id = tokenizer.pad_token_id

model.load_state_dict(torch.load('/content/drive/MyDrive/연세_수업자료/2학기/텍스트이해와인공지능/ddd_kcbert_base_e.pt', map_location=device))
model.to(device)

model.eval()
res = []
with torch.no_grad():
    for ids, mask in tqdm(test_loader):
        ids = ids.to(device)
        mask = mask.to(device)

        output = model(ids, mask)
        output = output.logits
        res.extend(output.argmax(axis = 1).tolist())

df_test_pre['output'] = res
# f1-score:59.6068966

model.safetensors:   0%|          | 0.00/438M [00:00<?, ?B/s]

Some weights of BertForSequenceClassification were not initialized from the model checkpoint at beomi/kcbert-base and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
100%|██████████| 65/65 [00:14<00:00,  4.62it/s]


In [ ]:
df_test_pre

,Unnamed: 0.1,Unnamed: 0,id,input,output
0,0,0,nikluge-au-2022-test-000001,극좌는 이 비겁 자 층을 제대로 요리 할 줄 안다,1
1,1,1,nikluge-au-2022-test-000002,내가 진짜 올 해 안에 차 산다,0
2,2,2,nikluge-au-2022-test-000003,선거 때마다 불장난 하는 못 된 버릇 대손 배워가지고 그러고 까불어 대면 너 나중에...,1
3,3,3,nikluge-au-2022-test-000004,난 9년 도에 이미 세상은 망해서 선 한자 들은 이미 모두 하늘로 올라갔고 남은 우...,0
4,4,4,nikluge-au-2022-test-000005,피의자로 가는 싸가지 없는 쓰래 기,1
...,...,...,...,...,...
2067,2067,2067,nikluge-au-2022-test-002068,근데 비계 올 사람만 찍어,0
2068,2068,2068,nikluge-au-2022-test-002069,지 들 입맛대로 막 가네 미친,1
2069,2069,2069,nikluge-au-2022-test-002070,얘는 지가 이 걸 쓰면서 왜 이런 생각을 못 하지,1
2070,2070,2070,nikluge-au-2022-test-002071,알겟나요,0


In [ ]:
def jsonlload(fname):
    with open(fname, "r", encoding="utf-8") as f:
        lines = f.read().strip().split("\n")
        j_list = [json.loads(line) for line in lines]
    return j_list

test_df = jsonlload('/content/drive/MyDrive/연세_수업자료/2학기/텍스트이해와인공지능/기말과제/NIKL_AU_2023_COMPETITION_v1.0/nikluge-au-2022-test.jsonl')
len(test_df)

2072

In [ ]:
for i, p in enumerate(df_test_pre['output']):
    test_df[i]['output'] = p

In [ ]:
def jsonldump(j_list, fname):
    with open(fname, "w", encoding='utf-8') as f:
        for json_data in j_list:
            f.write(json.dumps(json_data, ensure_ascii=False)+'\n')
jsonldump(test_df, '/content/drive/MyDrive/연세_수업자료/2학기/텍스트이해와인공지능/기말과제/test22_kcbert.jsonl')

In [ ]:
import numpy as np
import pandas as pd
import os, sys, re, math, random, time, json, pickle, tqdm, gc
from collections import defaultdict
import itertools as it

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader

from transformers import AdamW, get_linear_schedule_with_warmup
from transformers.optimization import get_cosine_schedule_with_warmup
from transformers import AutoTokenizer, AutoModelForSequenceClassification

from sklearn.model_selection import StratifiedKFold, KFold
from sklearn.metrics import f1_score, accuracy_score
from sklearn import preprocessing
from sklearn.model_selection import train_test_split
from tqdm import tqdm

class args:
    # amp = True
    gpu = '0'

    label_size = 2
    epochs=10
    batch_size=32
    weight_decay=1e-6
    max_len = 60

    start_lr = 2e-5

    num_workers=0
    seed=2022

os.environ["CUDA_VISIBLE_DEVICES"] = args.gpu
device = torch.device(f"cuda" if torch.cuda.is_available() else "cpu")
print(device)
##----------------
def set_seeds(seed=42):
    random.seed(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

set_seeds(seed=args.seed)

path = '/content/drive/MyDrive/연세_수업자료/2학기/텍스트이해와인공지능/'
model_name = "beomi/kcbert-base"

df_test = pd.read_csv('/content/drive/MyDrive/연세_수업자료/2학기/텍스트이해와인공지능/기말과제/df_test.csv', encoding = 'utf-8')
df_test_pre = pd.read_csv('/content/drive/MyDrive/연세_수업자료/2학기/텍스트이해와인공지능/기말과제/테스트 데이터-스마일게이트/df_test.csv', encoding = 'utf-8')
df_test_pre.dropna(inplace = True)

cuda


In [ ]:
from tokenizers.processors import TemplateProcessing

class TestDataset(Dataset):
    def __init__(self, dataset, text, label, tokenizer, max_len):
        self.dataset = dataset
        self.text = text
        self.label = label
        self.tokenizer = tokenizer
        self.max_len = max_len

    def __getitem__(self, idx):
        text = self.dataset.iloc[idx]['input']
        label = self.dataset.iloc[idx]['output']

        inputs = self.tokenizer(text,
                                return_tensors='pt',
                                truncation=True,
                                max_length=self.max_len,
                                padding='max_length',
                                add_special_tokens=True)

        input_ids = inputs['input_ids'][0]
        attention_mask = inputs['attention_mask'][0]

        return input_ids, attention_mask, label

    def __len__(self):
        return len(self.dataset)

model_name = "beomi/kcbert-base"
tokenizer = AutoTokenizer.from_pretrained(model_name)

test_dataset = TestDataset(df_test_pre, 'input', 'output', tokenizer, args.max_len)
test_loader = DataLoader(test_dataset, batch_size=args.batch_size, shuffle=True)

In [ ]:
model = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=args.label_size)
model.config.pad_token_id = tokenizer.pad_token_id

model.load_state_dict(torch.load('/content/drive/MyDrive/연세_수업자료/2학기/텍스트이해와인공지능/ddd_kcbert_base_e.pt', map_location=device))
model.to(device)

def evaluate_model(model, data_loader):
    target_lst = []
    pred_lst = []
    model.eval()
    with torch.no_grad():
        for ids, mask, target in tqdm(data_loader):
            ids = ids.to(device)
            mask = mask.to(device)
            target = target.to(device)

            output = model(ids, mask)
            pred_lst.extend(output.logits.argmax(dim=1).tolist())
            target_lst.extend(target.cpu().numpy())

    eval_score = f1_score(y_true=target_lst, y_pred=pred_lst, average='macro')
    eval_acc = accuracy_score(y_true=target_lst, y_pred=pred_lst)

    return eval_score, eval_acc

f1_score, accuracy = evaluate_model(model, test_loader)
print(f"F1-Score: {f1_score}, Accuracy: {accuracy}")

Some weights of BertForSequenceClassification were not initialized from the model checkpoint at beomi/kcbert-base and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
100%|██████████| 66/66 [00:07<00:00,  8.95it/s]

F1-Score: 0.7943090844490139, Accuracy: 0.7950310559006211
